<a href="https://colab.research.google.com/github/Aysha2004/S7_INTERNSHIP/blob/main/S7_Internship_d7_RNN_with_negation_words.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd

df = pd.read_csv('/content/IMDB Dataset.csv', encoding = 'latin1')

df['Sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

print(df.head())
print(df['Sentiment'].value_counts())

                                              review sentiment  Sentiment
0  One of the other reviewers has mentioned that ...  positive          1
1  A wonderful little production. <br /><br />The...  positive          1
2  I thought this was a wonderful way to spend ti...  positive          1
3  Basically there's a family where a little boy ...  negative          0
4  Petter Mattei's "Love in the Time of Money" is...  positive          1
Sentiment
1    25000
0    25000
Name: count, dtype: int64


In [5]:
df['Sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

In [8]:
import re

negation_words = [
    'not good', 'not bad', 'not great', "don't like", "didn't like",
    "never liked", "wasn't good", "isn't good", 'no good'
    ]

def clean_text(text):
  text = text.lower()

  text = re.sub(r"[^a-zA-Z\s']", " ", text)

  for phrase in negation_words:
    text = text.replace(phrase, phrase.replace(" ","_"))

  return text

In [9]:
df['review'] = df['review'].apply(clean_text)

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df['review'],
    df['Sentiment'],
    test_size=0.2,
    random_state=42
    )

In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 20000
max_len = 250

tokenizer = Tokenizer(num_words = vocab_size, oov_token = '<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen = max_len, padding = 'post')
X_test_pad = pad_sequences(X_test_seq, maxlen = max_len, padding = 'post')

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
      Embedding(vocab_size, 128, input_length = max_len),

      LSTM(128),

      Dense(64, activation = 'relu'),
      Dropout(0.5),

      Dense(1, activation = 'sigmoid')
])

model.compile(
    optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
model.compile(
    optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [20]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs = 5,
    validation_split = 0.2,
    batch_size = 64
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 331s 657ms/step - accuracy: 0.5377 - loss: 0.6783 - val_accuracy: 0.5599 - val_loss: 0.6600
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 310s 621ms/step - accuracy: 0.5998 - loss: 0.6165 - val_accuracy: 0.5918 - val_loss: 0.6261
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 326s 652ms/step - accuracy: 0.6762 - loss: 0.5345 - val_accuracy: 0.8397 - val_loss: 0.3985
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 308s 616ms/step - accuracy: 0.8947 - loss: 0.2777 - val_accuracy: 0.8621 - val_loss: 0.3840
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 310s 620ms/step - accuracy: 0.9433 - loss: 0.1656 - val_accuracy: 0.8715 - val_loss: 0.3411


In [21]:
loss, acc = model.evaluate(X_test_pad, y_test)
print(f"Test Loss: ",loss)
print(f"Test Accuracy: ", acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 46s 145ms/step - accuracy: 0.8780 - loss: 0.3250
Test Loss:  0.3250456154346466
Test Accuracy:  0.878000020980835


In [22]:
def predict_sentiment(review):
  review = clean_text(review)
  sequence = tokenizer.texts_to_sequences([review])
  padded_sequence = pad_sequences(sequence, maxlen = max_len, padding = 'post')
  prediction = model.predict(padded_sequence)[0][0]
  print("\nReview:" ,review)
  print("Score:",prediction)

  if prediction >= 0.5:
    print("Sentiment: Positive")
  else:
    print("Sentiment: Negative")


In [23]:
predict_sentiment("This movie was absolutely amazing")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 264ms/step

Review: this movie was absolutely amazing
Score: 0.7830244
Sentiment: Positive


In [24]:
predict_sentiment("This movie was an absolute flop")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 306ms/step

Review: this movie was an absolute flop
Score: 0.24299829
Sentiment: Negative
